# Notebook 16 — Cross-Validation: Zero-Shot Transfer Across All DO Gauges

Tests whether zero-shot transfer works across all training source gauges,
not just gauge 2473. Leave-one-out design: train on each gauge, transfer to all others.

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

REPO_ROOT = Path('/storage/homefs/tn20y076/AareML')
sys.path.insert(0, str(REPO_ROOT))

from src.config import (FEATURES, TARGETS, LOOKBACK, HORIZON,
                         TRAIN_END, VAL_END, SEED, RESULTS_DIR, FIGURES_DIR)
from src.data import load_gauge, preprocess, train_val_test_split, make_windows
from src.model import Seq2SeqLSTM, RiverDataset, train_model, predict, get_y_true
from src.metrics import mean_rmse, nse

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Hyperparameters (same as nb03)
HIDDEN, N_LAYERS, DROPOUT = 64, 2, 0.2   # lighter model for LOO speed
LR, EPOCHS, PATIENCE = 1e-3, 30, 5  # reduced for 4h budget

# All 16 DO gauges sorted by DO coverage
DO_GAUGES = [2473, 2009, 2613, 2143, 2016, 2174, 2415,
             2044, 2410, 2085, 2462, 2018, 2243, 2068, 2135, 2130]
print(f'DO gauges: {len(DO_GAUGES)}')

## 1. Leave-One-Out Cross-Validation

In [ ]:
from sklearn.preprocessing import StandardScaler

print('Loading and scaling all gauge data...')
gauge_data = {}

for gid in tqdm(DO_GAUGES, desc='Loading gauges'):
    try:
        raw  = load_gauge(gid)
        data = preprocess(raw)
        train_df, val_df, test_df = train_val_test_split(data)
        train_means = (pd.concat([train_df[FEATURES].mean(), train_df[TARGETS].mean()])
                       .groupby(level=0).first())

        # Build windows
        X_tr, y_tr, _ = make_windows(train_df, train_means, features=FEATURES, targets=TARGETS)
        X_va, y_va, _ = make_windows(val_df,   train_means, features=FEATURES, targets=TARGETS)
        X_te, y_te, _ = make_windows(test_df,  train_means, features=FEATURES, targets=TARGETS)

        if len(X_te) < 10:
            print(f'  {gid}: skipped (only {len(X_te)} test windows)')
            continue

        # Fit scalers on training data
        N_tr, L, F = X_tr.shape
        N_va = X_va.shape[0]
        N_te = X_te.shape[0]
        H    = y_tr.shape[1]
        T    = y_tr.shape[2]

        feat_sc = StandardScaler().fit(X_tr.reshape(-1, F))
        tgt_sc  = StandardScaler().fit(y_tr.reshape(-1, T))

        def scale(X_raw, y_raw, Ns):
            Xs = feat_sc.transform(X_raw.reshape(-1, F)).reshape(Ns, L, F).astype('float32')
            ys = tgt_sc.transform(y_raw.reshape(-1, T)).reshape(Ns, H, T).astype('float32')
            return Xs, ys

        Xs_tr, ys_tr = scale(X_tr, y_tr, N_tr)
        Xs_va, ys_va = scale(X_va, y_va, N_va)
        Xs_te, ys_te = scale(X_te, y_te, N_te)

        gauge_data[gid] = {
            'ds_tr': RiverDataset(Xs_tr, ys_tr),
            'ds_va': RiverDataset(Xs_va, ys_va),
            'ds_te': RiverDataset(Xs_te, ys_te),
            'tgt_sc': tgt_sc,
            'y_te_raw': y_te,   # unscaled, for RMSE in original units
        }
        print(f'  {gid}: {N_tr} train / {N_va} val / {N_te} test windows')
    except Exception as e:
        print(f'  {gid}: FAILED ({e})')

print(f'\nLoaded: {len(gauge_data)}/{len(DO_GAUGES)} gauges')


In [ ]:
DO_IDX = list(TARGETS).index('O2C_sensor')
results = []

for src_gid in tqdm(DO_GAUGES, desc='Source gauges'):
    if src_gid not in gauge_data:
        continue

    gd = gauge_data[src_gid]
    dl_tr = torch.utils.data.DataLoader(gd['ds_tr'], batch_size=256, shuffle=True)
    dl_va = torch.utils.data.DataLoader(gd['ds_va'], batch_size=256, shuffle=False)

    model = Seq2SeqLSTM(
        n_feat=len(FEATURES), n_tgt=len(TARGETS),
        hidden=HIDDEN, n_layers=N_LAYERS, dropout=DROPOUT,
    ).to(DEVICE)
    model, _ = train_model(model, dl_tr, dl_va, device=DEVICE,
                           epochs=EPOCHS, lr=LR, patience=PATIENCE, verbose=False)

    # Zero-shot transfer to all other gauges
    for tgt_gid in DO_GAUGES:
        if tgt_gid == src_gid or tgt_gid not in gauge_data:
            continue
        tgt = gauge_data[tgt_gid]
        dl_te = torch.utils.data.DataLoader(tgt['ds_te'], batch_size=256, shuffle=False)

        try:
            # Predict (scaled) then inverse transform
            y_pred_sc = predict(model, dl_te, DEVICE)           # shape (N, H, T)
            y_pred = tgt['tgt_sc'].inverse_transform(
                y_pred_sc.reshape(-1, len(TARGETS))
            ).reshape(y_pred_sc.shape)

            y_true = tgt['y_te_raw']   # already in original units

            # DO only
            y_pred_do = y_pred[:, :, DO_IDX]
            y_true_do = y_true[:, :, DO_IDX]

            rmse_val = float(np.sqrt(np.mean((y_pred_do - y_true_do)**2)))
            ss_res   = np.sum((y_true_do - y_pred_do)**2)
            ss_tot   = np.sum((y_true_do - y_true_do.mean())**2)
            nse_val  = float(1 - ss_res/ss_tot) if ss_tot > 0 else float('nan')

            results.append({
                'source_gauge': src_gid,
                'target_gauge': tgt_gid,
                'rmse_do': round(rmse_val, 4),
                'nse_do':  round(nse_val,  3),
            })
        except Exception as e:
            import traceback
            print(f'  {src_gid}→{tgt_gid}: {e}')
            traceback.print_exc()

print(f'\nResults collected: {len(results)} pairs')
df_cv = pd.DataFrame(results) if results else pd.DataFrame(columns=['source_gauge','target_gauge','rmse_do','nse_do'])
src_2473 = df_cv[df_cv['source_gauge'] == 2473]
src_other = df_cv[df_cv['source_gauge'] != 2473]

print(f'Total pairs:    {len(df_cv)}')
print(f'Overall RMSE:   {df_cv["rmse_do"].mean():.4f} mg/L')
print(f'Overall NSE:    {df_cv["nse_do"].mean():.3f}')
print(f'Source=2473:    RMSE={src_2473["rmse_do"].mean():.4f}  NSE={src_2473["nse_do"].mean():.3f}')
print(f'Other sources:  RMSE={src_other["rmse_do"].mean():.4f}  NSE={src_other["nse_do"].mean():.3f}')

df_cv.to_csv(RESULTS_DIR / 'cv_transfer_results.csv', index=False)
print(f'Saved: {RESULTS_DIR}/cv_transfer_results.csv')


## 2. Results Heatmap

In [ ]:
if df_cv.empty:
    print("No results to plot — check errors above")
else:
    # Pivot to matrix
    pivot = df_cv.pivot(index='source_gauge', columns='target_gauge', values='rmse_do')
    
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn_r',
                vmin=0.2, vmax=1.5, ax=ax,
                cbar_kws={'label': 'RMSE (mg/L DO)'})
    ax.set_title('Zero-Shot Transfer RMSE: Source \u2192 Target Gauge\n(lower = better transfer)',
                 fontsize=13)
    ax.set_xlabel('Target Gauge'); ax.set_ylabel('Source Gauge')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'nb16_cv_heatmap.png', dpi=150)
    plt.close()
    print('Saved: nb16_cv_heatmap.png')

## 3. Per-Source Summary

In [ ]:
if df_cv.empty:
    print("No results to plot")
else:
    src_summary = df_cv.groupby('source_gauge')['rmse_do'].mean().sort_values()
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#01696F' if g == 2473 else '#D4D1CA' for g in src_summary.index]
    ax.bar(src_summary.index.astype(str), src_summary.values, color=colors)
    ax.axhline(src_summary.mean(), color='#d62728', linestyle='--', linewidth=1.5,
               label=f'Mean across all sources: {src_summary.mean():.3f} mg/L')
    ax.set_xlabel('Source Gauge'); ax.set_ylabel('Mean Transfer RMSE (mg/L)')
    ax.set_title('Mean Zero-Shot Transfer RMSE by Source Gauge\n(teal = gauge 2473, our chosen training gauge)')
    ax.legend(); plt.xticks(rotation=45); plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'nb16_source_comparison.png', dpi=150)
    plt.close()
    print('Saved: nb16_source_comparison.png')

## 4. Summary

In [ ]:
if df_cv.empty:
    print("No results — check cell 4 errors")
else:
    print('=' * 60)
    print('CROSS-VALIDATION SUMMARY')
    print('=' * 60)
    print(f'Total transfer pairs:     {len(df_cv)}')
    print(f'Overall mean RMSE:        {df_cv["rmse_do"].mean():.4f} mg/L')
    print(f'Overall mean NSE:         {df_cv["nse_do"].mean():.3f}')
    print(f'')
    print(f'Source=2473 mean RMSE:    {src_2473["rmse_do"].mean():.4f} mg/L')
    print(f'All-source mean RMSE:     {df_cv["rmse_do"].mean():.4f} mg/L')
    print(f'')
    print(f'Best source gauge:        {src_summary.index[0]} (RMSE={src_summary.iloc[0]:.4f})')
    print(f'Worst source gauge:       {src_summary.index[-1]} (RMSE={src_summary.iloc[-1]:.4f})')
    print(f'')
    print(f'Interpretation: {"gauge 2473 transfers better than average" if src_2473["rmse_do"].mean() < df_cv["rmse_do"].mean() else "gauge 2473 is representative of average transfer performance"}')